# Valutazione del modello su GTZAN

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alessandro1040/bpm-crnn/blob/main/notebooks/02_valutazione_gtzan.ipynb)

Riproduce (o misura su un sottoinsieme) le metriche del modello rilasciato sul
**test set** di GTZAN: 100 tracce tenute fuori dall'addestramento, split
raggruppato per hash audio (nessun leak).

Cosa fa:
0. dipendenze;
1. clone del repo + pesi da GitHub;
2. scarica GTZAN (~1.2 GB) + le annotazioni di tempo (`gtzan_tempo_beat`);
3. costruisce metadati + split + cache Mel (float16, ~330 MB);
4. valuta con `src.evaluate`: metriche complete per i 3 output (mean/frame/song),
   tabella per genere, assegnazione d'ottava e grafici;
5. confronta con i numeri committati in `results/gtzan/test_metrics.json`.

`N_PER_GENRE = None` scarica e valuta tutto (riproduce il README); un intero,
es. `12`, valuta solo 12 tracce per genere: molto piu' veloce ma lo split
cambia (numeri indicativi, non confrontabili con il README).

In [ ]:
#@title 0. Dipendenze e import
%pip install -q torch librosa soundfile pandas matplotlib tqdm requests

import json
import subprocess
import sys
import tarfile
import urllib.request
from pathlib import Path

import numpy as np
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)

In [ ]:
#@title 1. Clona il repo e scarica i pesi
import json, os, subprocess, sys, urllib.request
from pathlib import Path

import numpy as np
import torch

REPO_URL = "https://github.com/Alessandro1040/bpm-crnn.git"
ROOT = Path("/content/bpm-crnn") if Path("/content").is_dir() else Path.cwd() / "bpm-crnn"
if not (ROOT / "src" / "infer.py").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(ROOT)], check=True)
sys.path.insert(0, str(ROOT))
print("repo:", ROOT)

CKPT = ROOT / "models" / "bpm_crnn_gtzan.pt"
RAW_URL = "https://raw.githubusercontent.com/Alessandro1040/bpm-crnn/main/models/bpm_crnn_gtzan.pt"
RELEASE_URL = "https://github.com/Alessandro1040/bpm-crnn/releases/download/v1.0-gtzan/best.pth"
if not CKPT.exists():
    try:
        print("scarico i pesi da GitHub (models/bpm_crnn_gtzan.pt, 21 MB) ...")
        urllib.request.urlretrieve(RAW_URL, CKPT)
    except Exception as exc:                       # fallback: release asset (64 MB, con optimizer)
        print("fallback sulla release:", exc)
        urllib.request.urlretrieve(RELEASE_URL, CKPT)
print("pesi:", CKPT, f"{CKPT.stat().st_size/1e6:.1f} MB")

In [ ]:
#@title 2. Scarica GTZAN e le annotazioni di tempo
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)
TAR = DATA / "genres.tar.gz"
AUDIO = DATA / "genres"
ANNOT = DATA / "gtzan_tempo_beat" / "tempo"

if not AUDIO.is_dir():
    if not TAR.exists():
        print("scarico GTZAN (~1.2 GB) ...")
        urllib.request.urlretrieve(
            "https://huggingface.co/datasets/marsyas/gtzan/resolve/main/data/genres.tar.gz",
            TAR)
    print("estraggo ...")
    with tarfile.open(TAR, "r:gz") as tar:
        tar.extractall(DATA)
    TAR.unlink(missing_ok=True)          # libera 1.2 GB
if not ANNOT.is_dir():
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/TempoBeatDownbeat/gtzan_tempo_beat.git",
                    str(DATA / "gtzan_tempo_beat")], check=True)

print("traccia audio:", len(list(AUDIO.glob("*/*.wav"))))
print("annotazioni  :", len(list(ANNOT.glob("*.bpm"))))

In [ ]:
#@title 3. Metadati, split senza leakage e cache Mel
from src.dataset import build_metadata, cache_mels, make_splits

N_PER_GENRE = None          # None = tutte le tracce; es. 12 = sottoinsieme rapido

if N_PER_GENRE is None:
    audio_root = AUDIO
else:
    subset = DATA / "subset"
    if subset.exists():
        subprocess.run(["rm", "-rf", str(subset)], check=True)
    subset.mkdir(parents=True)
    for genre_dir in sorted(p for p in AUDIO.iterdir() if p.is_dir()):
        (subset / genre_dir.name).mkdir(exist_ok=True)
        for wav in sorted(genre_dir.glob("*.wav"))[:N_PER_GENRE]:
            (subset / genre_dir.name / wav.name).symlink_to(wav.resolve())
    audio_root = subset
    print("sottoinsieme:", len(list(subset.glob("*/*.wav"))), "tracce")

META = DATA / ("meta_subset.csv" if N_PER_GENRE else "meta.csv")
meta = build_metadata(DATA, audio_root=audio_root, annot_dir=ANNOT, out_csv=META)
meta = make_splits(meta, seed=42)
meta.to_csv(META, index=False)
cache_mels(meta, ROOT / ("data/cache_subset" if N_PER_GENRE else "data/cache"))

In [ ]:
#@title 4. Valutazione completa sul test set
CACHE_DIR = ROOT / ("data/cache_subset" if N_PER_GENRE else "data/cache")
OUT_DIR = ROOT / ("runs/eval_subset" if N_PER_GENRE else "runs/eval_full")
cmd = [sys.executable, "-m", "src.evaluate", "--ckpt", str(CKPT),
       "--cache-dir", str(CACHE_DIR), "--out-dir", str(OUT_DIR),
       "--plots-dir", str(ROOT / "assets"), "--batch-size", "32"]
print(" ".join(str(c) for c in cmd), "\n")
subprocess.run(cmd, cwd=ROOT, check=True)

In [ ]:
#@title 5. Confronto con i numeri committati nel README
from IPython.display import Image, display

display(Image(filename=str(ROOT / "assets" / "predictions_test.png")))

report = json.loads((OUT_DIR / "metrics_report_test.json").read_text())
ref_path = ROOT / "results" / "gtzan" / "test_metrics.json"
if ref_path.exists():
    ref = json.loads(ref_path.read_text())
    righe = ["MAE", "MedianAE", "Acc_1BPM", "Acc_1%", "Acc_5%",
             "Acc_octave_4%", "Octave_error_rate_4%", "P_score", "Cemgil"]
    print(f"{'metrica':<24}{'questo run':>12}{'README':>12}{'delta':>10}")
    print("-" * 58)
    for key in righe:
        now = report["metrics"]["mean"][key]
        was = ref[f"mean/{key}"]
        scala = 100.0 if key.startswith(("Acc", "Octave")) else 1.0
        print(f"{key:<24}{scala * now:12.3f}{scala * was:12.3f}"
              f"{scala * (now - was):+10.3f}")
    print("\n(I numeri del README sono con N_PER_GENRE=None e split seed 42: "
          "devono coincidere.)")
else:
    print("results/gtzan/test_metrics.json non trovato nel repo")